# Stage 1: MLflow 3 Evaluation Fundamentals
### Practical AgentOps: From PoC to Prod with MLflow 3 — ODSC AI East 2026

---

**The scenario:** You're an ML engineer at a startup building *Mise en Place*, a cooking assistant app. The PM shared a prototype with beta users and everyone likes it — now the pressure is on to turn it into something shippable. That means evaluation.

In this stage we build the simplest possible agent — a bare LLM with a system prompt — and use it as a canvas to learn MLflow 3's evaluation primitives from the ground up. We'll iterate through two rounds of product team feedback, discover the classic ways eval sets break, and finish with a scorer suite that actually captures what the team cares about.

By the end of this notebook you will have hands-on experience with:

- `mlflow.openai.autolog()` — automatic tracing of every LLM call
- `mlflow.genai.evaluate()` — running structured evaluations
- `expected_facts` vs `expected_response` — two flavours of ground truth
- Common ways eval sets go wrong — and how to fix them
- Built-in scorers: `Correctness`, `Safety`, `RelevanceToQuery`, `Guidelines`, `Groundedness`
- The `@scorer` decorator — writing deterministic custom scorers
- `make_judge()` — building LLM-as-a-judge evaluators

> **No API key?** You can read through every cell and understand the concepts. Cells that call the model are clearly marked — skip those and pick back up at the next markdown section.

---
## 0 — Environment Setup

1. Follow steps in the 'workshop/setup' folder
2. cmd/ctrl + shift + p
3. Python: Select Interpreter
4. Select your recently created environment

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import mlflow

load_dotenv()  # reads GEMINI_API_KEY from .env


print(f"MLflow version: {mlflow.__version__}")
print(f"GEMINI_API_KEY present: {os.getenv('GEMINI_API_KEY') != ""}")

MLflow version: 3.11.1
GEMINI_API_KEY present: True


### Configuring the OpenAI client to point at Gemini

We use the **OpenAI SDK** pointed at Google's OpenAI-compatible endpoint. This is an increasingly common pattern — the SDK is a familiar interface, and swapping providers is just a matter of changing `base_url` and the API key.

| Provider | `base_url` | Key env var |
|---|---|---|
| OpenAI (default) | *(omit)* | `OPENAI_API_KEY` |
| **Gemini** ✓ | `https://generativelanguage.googleapis.com/v1beta/openai/` | `GEMINI_API_KEY` |
| Azure OpenAI | `https://<resource>.openai.azure.com/` | `AZURE_OPENAI_API_KEY` |
| Ollama (local) | `http://localhost:11434/v1` | *(none required)* |

In [3]:
client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.1-flash-lite-preview"

---
## 1 — Autologging and the Experiment

Two lines are all it takes to get full tracing into the MLflow UI.

- `mlflow.set_experiment()` — creates (or resumes) a named experiment bucket for all runs
- `mlflow.openai.autolog()` — patches the OpenAI SDK to emit a trace for every `chat.completions.create()` call, capturing inputs, outputs, token counts, and latency automatically

Once these are set, **every call in this notebook is captured with no extra code**.

In [7]:
tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5000")
mlflow.set_tracking_uri(tracking_uri)

mlflow.set_experiment("recipe-assistant")
mlflow.openai.autolog()

print("Autologging enabled. Open the MLflow UI to follow along:")
print("  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000")
print("If previous instance is running, run in terminal: kill -9 $(lsof -t -i:3000)")

2026/04/13 10:24:49 INFO mlflow.tracking.fluent: Experiment with name 'recipe-assistant' does not exist. Creating a new experiment.


Autologging enabled. Open the MLflow UI to follow along:
  mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
If previous instance is running, run in terminal: kill -9 $(lsof -t -i:3000)


---
## 2 — Your First Completions Call

Before we think about evaluation, structure, or scaffolding — let's just build the thing. A system prompt and a single completions call. This is the PoC.

In [10]:
SYSTEM_PROMPT = """You are a helpful cooking assistant. Answer questions about recipes,
cooking techniques, ingredients, and meal planning. Be practical and clear."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": "How do I make perfectly scrambled eggs?"},
    ],
    temperature=0.1,
    max_completion_tokens=512,
)

print(response.choices[0].message.content)

Making "perfect" scrambled eggs is a matter of technique rather than a complex recipe. The goal is to achieve a soft, creamy, custard-like texture rather than dry, rubbery curds.

Here is the professional method for the best scrambled eggs:

### 1. The Prep
*   **Use a non-stick pan:** This is non-negotiable for easy cleanup and even cooking.
*   **Whisk thoroughly:** Crack 2–3 eggs into a bowl and whisk them vigorously with a fork or whisk until the yolks and whites are completely combined and no streaks remain. 
*   **Seasoning:** Add a pinch of salt *before* cooking. Contrary to old myths, salt does not toughen the eggs if you cook them immediately; it helps break down the proteins for a more tender result.

### 2. The Heat
*   **Low and slow is key:** Place your pan over **medium-low heat**. If the pan is too hot, the eggs will seize up and become rubbery before you have a chance to stir them.
*   **Butter:** Add about a tablespoon of unsalted butter. Let it melt until it’s foaming

Trace(trace_id=tr-e49143de16f1625a757db9cd488b4429)

### Ship it! ...right?

That works great. You show it to your product team and they have lots of questions. How does it handle recipes? Is it accurate? Is it ready for user testing?

> **Sam (Product):** "How do we *know* it's good? Like, do we have any metrics? What if it gives someone bad food safety advice? What if the next model update makes it worse? Have you discussed anything with legal?"

You don't have an answer yet. Time to think about **evaluation**.

---
## 3 — Pick a Metric(s): Pre-built Scorers and Judges

Before we write our own evaluation logic, let's see what MLflow gives us out of the box. `mlflow.genai.evaluate()` accepts a list of **scorers** — each one measures a different quality dimension of your agent's output.

### Built-in Scorers

| Scorer | What it measures | Needs ground truth? | Best for |
|---|---|---|---|
| `Correctness()` | Does the response match the expected answer? | Yes (`expected_facts` or `expected_response`) | Factual accuracy — recipes, ingredients, temperatures |
| `RelevanceToQuery()` | Is the response actually about what was asked? | No | Catching off-topic or generic answers |
| `Safety()` | Does the response avoid harmful content? | No | Food safety, allergy advice, dangerous instructions |
| `Guidelines(name, guidelines)` | Does the response follow a custom rule? | No | Product requirements — tone, format, measurements |
| `Groundedness()` | Is the response supported by the provided context? | Yes (requires `context`) | RAG applications (not needed yet, but good to know) |

### The `model` parameter — which LLM runs the judge?

Every built-in scorer and `make_judge()` call uses an LLM under the hood to evaluate your agent's output. By default this is **`openai:/gpt-4o-mini`** — so you need an `OPENAI_API_KEY` set even if your agent itself uses a different provider.

The `model` parameter uses a `"provider:/model-name"` format:

```python
from mlflow.genai.scorers import Correctness, Guidelines

# Default — uses openai:/gpt-4o-mini (requires OPENAI_API_KEY)
Correctness()

# Use a different OpenAI model
Correctness(model="openai:/gpt-4o")

# Use Anthropic as the judge
Guidelines(model="anthropic:/claude-sonnet-4", name="tone", guidelines="...")

# Use Google as the judge
Correctness(model="google:/gemini-2.0-flash")
```

Same syntax works for `make_judge()`:

```python
from mlflow.genai.judges import make_judge

my_judge = make_judge(
    name="my_judge",
    instructions="Evaluate {{ outputs }} for ...",
    model="openai:/gpt-4o",        # override the judge model
    feedback_value_type=bool,
)
```

**Supported providers:** `openai:/`, `anthropic:/`, `google:/`, and any other [LiteLLM-supported provider](https://docs.litellm.ai/docs/providers).

> **For this workshop** we'll use the default (`openai:/gpt-4o-mini`). If you only have a Gemini key, you can pass `model="google:/gemini-2.0-flash"` to every scorer. Just keep in mind: your *agent* model and your *judge* model don't have to be the same — and often shouldn't be.

### LLM-as-a-Judge with `make_judge()`

For nuanced requirements that don't fit a single guideline sentence, `make_judge()` lets you write a full rubric:

```python
from mlflow.genai.judges import make_judge

my_judge = make_judge(
    name="my_custom_judge",
    instructions="Evaluate {{ outputs }} for ...",
    feedback_value_type=bool,  # or float for a 0-1 score
)
```

### Custom Deterministic Scorers with `@scorer`

For rules that can be checked with code (regex, length, format), the `@scorer` decorator is faster and free — **no judge model needed**:

```python
from mlflow.genai.scorers import scorer

@scorer
def has_measurement(outputs: str, **kwargs) -> bool:
    return bool(re.search(r'\d+\s*(cup|tbsp|tsp|oz|g|ml|min)', outputs, re.I))
```

### When to use which?

| | `Correctness` / `Guidelines` | `make_judge()` | `@scorer` |
|---|---|---|---|
| **Speed** | ~1 LLM call per row | ~1 LLM call per row | Instant (pure Python) |
| **Cost** | Tokens per eval | Tokens per eval | Free |
| **Judge model** | Configurable via `model=` | Configurable via `model=` | N/A — no LLM |
| **Flexibility** | Pre-built patterns | Full custom rubric | Code-expressible rules only |
| **Use when** | Standard quality checks | Nuanced, context-dependent rules | Format, length, pattern checks |

> **Our plan:** Start with `Correctness` to check factual accuracy, then layer on more scorers as product requirements emerge.

---
## 4 — Building the Evaluation Dataset

### The `predict_fn` contract

`mlflow.genai.evaluate()` needs a function with a specific signature:

```python
def predict_fn(inputs: dict) -> str:
```

- **Takes** a `dict` (one row's `inputs` from the dataset)
- **Returns** a `str` (the agent's response)

So we need to wrap our completions call into a function first.

### Dataset format

Each row in the eval dataset is a dict with:
- `inputs` — the arguments passed to your `predict_fn`
- `expectations` — the ground truth used by scorers

Two flavours of ground truth:

| Key | What it means | Best for |
|---|---|---|
| `expected_facts` | Concepts that must appear (semantically) in the response | Factual questions with clear right answers |
| `expected_response` | A full reference answer the response is compared against holistically | Open-ended or technique questions |

In [ ]:
# Wrap our completions call into the predict_fn contract
def recipe_agent(inputs: dict) -> str:
    """Bare recipe assistant — no tools, no retrieval."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": inputs["question"]},
        ],
        temperature=0.1,
        max_completion_tokens=512,
    )
    return response.choices[0].message.content


# Our first evaluation dataset
eval_dataset_v1 = [
    {
        "inputs": {"question": "How do I make guacamole?"},
        "expectations": {
            "expected_facts": ["avocado", "lime", "salt", "onion", "cilantro"]
        },
    },
    {
        "inputs": {"question": "What can I use instead of eggs in baking?"},
        "expectations": {
            "expected_facts": ["applesauce", "banana", "flax"]
        },
    },
    {
        "inputs": {"question": "How do I know when chicken breast is safely cooked?"},
        "expectations": {
            "expected_facts": ["165", "internal temperature", "thermometer"]
        },
    },
    {
        # ⚠️ BUG: This uses expected_facts for an open-ended technique question.
        # The facts are too specific — the agent might describe the same process
        # with different words and still be correct. We'll fix this later.
        "inputs": {"question": "How do I make pan gravy from drippings?"},
        "expectations": {
            "expected_facts": [
                "fond",             # agent might say "brown bits" or "drippings"
                "deglaze",          # agent might say "add liquid and scrape"
                "whisk in flour",   # agent might use a different thickener
                "simmer 5 minutes", # too specific on time
            ]
        },
    },
]

print(f"Eval set size: {len(eval_dataset_v1)} examples")

---
## 5 — First Evaluation Run: Correctness Only

We start with just `Correctness` — an LLM-as-a-judge scorer that checks whether the response satisfies the ground truth expectations.

`mlflow.genai.evaluate()` will:
1. Call `predict_fn` (our `recipe_agent`) on every row in `data`
2. Run each scorer against the output
3. Log everything as a new MLflow run (visible in the UI under `recipe-assistant`)
4. Return a results object with per-row scores and aggregate metrics

> **Watch for it:** The pan gravy row has intentionally over-specified `expected_facts`. See if `Correctness` flags it.

In [ ]:
from mlflow.genai.scorers import Correctness

results_v1 = mlflow.genai.evaluate(
    data=eval_dataset_v1,
    predict_fn=recipe_agent,
    scorers=[Correctness()],
)

print("=== Run 1: Correctness Only ===")
print(results_v1.metrics)

In [ ]:
# Per-row scores — look at which questions scored lowest
results_v1.tables["eval_results_table"]

---
## 6 — Common Eval Trap: Brittle Expected Facts

Did the pan gravy row score lower than you expected? That's the **naming variation problem** in action.

Our `expected_facts` demanded `"fond"` — but the agent might correctly say "brown bits at the bottom of the pan." We asked for `"deglaze"` — the agent might say "pour in some stock and scrape up the bits." Both are correct cooking advice, but the too-literal facts can confuse the scorer.

This is one of the most common pitfalls in eval: **your ground truth uses one word, the model uses another, and a perfectly good answer gets penalized.**

Let's make it even more obvious with an extreme example.

In [ ]:
# Over-specified: demands exact culinary terminology the agent may not use
brittle_example = {
    "inputs": {"question": "How do I make pan gravy from drippings?"},
    "expectations": {
        "expected_facts": [
            "fond",                     # agent might say "brown bits" or "drippings stuck to the pan"
            "deglaze with wine",        # agent might use stock, or just say "add liquid"
            "whisk in flour",           # agent might say "stir in flour" or use cornstarch
            "simmer exactly 5 minutes", # too specific — agent might say "a few minutes"
            "strain through chinois",   # most home cooks don't own a chinois!
        ]
    },
}

results_brittle = mlflow.genai.evaluate(
    data=[brittle_example],
    predict_fn=recipe_agent,
    scorers=[Correctness()],
)

print("=== Brittle expectations demo ===")
results_brittle.tables["eval_results_table"]

### The Fix: Write expectations at the right level of specificity

Good `expected_facts` describe **what the answer must convey**, not the exact word it must use.

A useful rule of thumb: *would a home cook consider the response correct?* If yes, your expectations should pass it.

For technique questions with multiple valid approaches, `expected_response` is often a better fit than a long `expected_facts` list — it lets the LLM judge weigh the answer holistically rather than doing term-by-term matching.

Let's fix our pan gravy row and re-run:

---
## 7 — 💬 Product Team Check-in #1

*You share the initial eval results with your product team. The Correctness scores are reasonable, but the team has been collecting user feedback and has some specific asks.*

---

> **Sam (Product):** "Hey — the beta testers really like the agent overall. But we're seeing a pattern in the feedback: users are getting answers like *'add some butter'* or *'cook until done'* and they don't know what to actually do. Can we make sure it gives real measurements and specific times?"
>
> **You:** "Makes sense. The system prompt doesn't ask for that. I can add a `Guidelines` scorer to catch vague answers, and update the eval set to include cases that require quantities."
>
> **Sam:** "Perfect. Also, two more things: we want responses to always mention roughly how long the recipe will take — even just 'about 30 minutes' — so users can plan. And our tone is a bit flat. Users said it feels like reading a manual. Can we be more conversational?"
>
> **You:** "All three of those are scoreable. I'll add `Guidelines` scorers for measurements and cook time, and a tone guideline. The dataset needs a few new examples too — right now we don't have anything that would catch a 'no cook time' failure."

---

Let's translate that conversation into scorers and updated eval data.

In [ ]:
eval_dataset_v2 = [
    {
        "inputs": {"question": "How do I make guacamole?"},
        "expectations": {
            "expected_facts": ["avocado", "lime", "salt", "onion", "cilantro"]
        },
    },
    {
        "inputs": {"question": "What can I use instead of eggs in baking?"},
        "expectations": {
            "expected_facts": ["applesauce", "banana", "flax"]
        },
    },
    {
        "inputs": {"question": "How do I know when chicken breast is safely cooked?"},
        "expectations": {
            "expected_facts": ["165", "internal temperature", "thermometer"]
        },
    },
    {
        # Fixed: switched from brittle expected_facts to a holistic expected_response
        "inputs": {"question": "How do I make pan gravy from drippings?"},
        "expectations": {
            "expected_response": (
                "After cooking meat, keep the drippings and browned bits in the pan. "
                "Place over medium heat, sprinkle in flour and stir to combine with the fat. "
                "Gradually add stock or broth while stirring, scraping up the flavorful bits "
                "from the bottom. Simmer for a few minutes until thickened to your liking."
            )
        },
    },
    # ── New cases that will catch the failures Sam flagged ──────────────────
    {
        # Should mention specific quantities, not just "some onions"
        "inputs": {"question": "How do I caramelize onions?"},
        "expectations": {
            "expected_facts": ["low heat", "30", "butter"]
        },
    },
    {
        # Should give a concrete time estimate
        "inputs": {"question": "How long does it take to make homemade pizza dough?"},
        "expectations": {
            "expected_facts": ["rise", "hour"]
        },
    },
]

print(f"Eval set size: {len(eval_dataset_v2)} examples")

---
## 8 — Adding the Scorers Sam Asked For

Now we put the pre-built scorers from Section 3 to work. `Guidelines` is the workhorse — it's how we encode Sam's product requirements as evaluatable constraints.

In [ ]:
from mlflow.genai.scorers import Correctness, Safety, RelevanceToQuery, Guidelines

results_v3 = mlflow.genai.evaluate(
    data=eval_dataset_v2,
    predict_fn=recipe_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="uses_specific_measurements",
            guidelines=(
                "When providing a recipe or cooking instructions, the response must include "
                "specific measurements (e.g. '2 tablespoons', '1 cup', '200g') rather than "
                "vague quantities like 'some', 'a handful', or 'a bit of'."
            ),
        ),
        Guidelines(
            name="includes_time_estimate",
            guidelines=(
                "Responses about recipes or cooking processes must mention how long the "
                "process takes — either a total time, a range, or per-step times."
            ),
        ),
        Guidelines(
            name="conversational_tone",
            guidelines=(
                "Responses should feel like advice from a knowledgeable friend, not a "
                "technical manual. Avoid overly formal language or bullet-point-only answers."
            ),
        ),
    ],
)

print("=== Run 2: Guidelines scorers added ===")
print(results_v3.metrics)

---
## 9 — Custom Scorers with `@scorer`

`Guidelines` scorers use an LLM judge, which makes them flexible but slow and non-deterministic. For rules that can be expressed as code, a `@scorer` function is faster, cheaper, and completely predictable.

Your function receives:
- `outputs` — the string returned by `predict_fn`
- `inputs` — the original input dict (keyword arg)
- `expectations` — the ground truth dict (keyword arg)

Return `bool`, `int`, `float`, or a `Feedback` object with a rationale.

In [ ]:
import re
from mlflow.genai.scorers import scorer, Feedback


@scorer
def has_measurement(outputs: str, **kwargs) -> bool:
    """
    Checks that the response includes at least one specific measurement.
    Matches patterns like: '2 cups', '1 tbsp', '200g', '350°F', '30 minutes'.
    This is a fast, deterministic check — no LLM call needed.
    """
    measurement_pattern = re.compile(
        r'\b\d+\.?\d*\s*'
        r'(cup|tbsp|tsp|tablespoon|teaspoon|oz|ounce|lb|pound|g|gram|kg|'
        r'ml|litre|liter|°[CF]|degree|minute|min|hour|hr)s?\b',
        re.IGNORECASE
    )
    return bool(measurement_pattern.search(outputs))


@scorer
def not_overwhelming(outputs: str, **kwargs) -> Feedback:
    """
    Checks the response isn't too long for a mobile cooking assistant.
    Returns a Feedback object so we can see the character count in the UI.
    """
    char_count = len(outputs)
    passes = char_count <= 1200
    return Feedback(
        value=passes,
        rationale=f"{char_count} characters — {'within' if passes else 'over'} the 1200-char limit",
    )


results_v4 = mlflow.genai.evaluate(
    data=eval_dataset_v2,
    predict_fn=recipe_agent,
    scorers=[
        Correctness(),
        RelevanceToQuery(),
        Guidelines(
            name="uses_specific_measurements",
            guidelines=(
                "When providing a recipe or cooking instructions, the response must include "
                "specific measurements rather than vague quantities like 'some' or 'a handful'."
            ),
        ),
        Guidelines(
            name="includes_time_estimate",
            guidelines=(
                "Responses about recipes or cooking processes must mention how long the process takes."
            ),
        ),
        has_measurement,        # fast deterministic check
        not_overwhelming,       # fast deterministic check with rationale
    ],
)

print("=== Run 3: Custom scorers added ===")
print(results_v4.metrics)

In [ ]:
# Look at the per-row breakdown — which question failed which scorer?
# Pay attention to whether has_measurement and Guidelines disagree.
# When a deterministic scorer and an LLM judge diverge, one of them is wrong —
# and figuring out which is a useful debugging exercise.
results_v4.tables["eval_results_table"]

---
## 10 — 💬 Product Team Check-in #2

*A week later, after iterating on the system prompt and sharing results with the team...*

---

> **Sam:** "The measurements and timing are much better — users are noticing. Two new asks from this sprint."
>
> **Sam:** "First: we keep hearing from users who are vegetarian or have dietary restrictions. Can the agent proactively mention substitutions? Like if someone asks for beef tacos, can it also say 'for a vegetarian version, try black beans or roasted sweet potato'?"
>
> **You:** "That's a behavior change — I need to update the system prompt and add eval cases that specifically test for it. Otherwise we'll ship it and have no way to know if it's working."
>
> **Sam:** "Agreed. Second thing — and this came from Legal — we need to be careful about food safety. Specifically allergy information. We can't have the agent confidently say 'you can substitute X for Y' when X might trigger a serious allergy. We need it to hedge appropriately."
>
> **You:** "That's exactly what the `Safety` scorer is for. I'll add allergy-adjacent test cases and turn Safety on. I'll also write a custom judge for the dietary substitution behavior since it's nuanced enough that a simple guideline might miss edge cases."

---

Let's add the new test cases and update the scorer suite.

In [ ]:
# First: update the system prompt to encourage dietary substitution suggestions
SYSTEM_PROMPT_V2 = """You are a helpful cooking assistant for Mise en Place, a cooking app.
Answer questions about recipes, cooking techniques, ingredients, and meal planning.

Guidelines:
- Always include specific measurements and time estimates
- Be conversational and encouraging, like a knowledgeable friend
- When giving recipes, briefly mention a vegetarian or dietary alternative if one exists
- For food safety questions, be accurate and include appropriate caveats"""


def recipe_agent_v2(inputs: dict) -> str:
    """Updated agent with dietary substitution and safety guidance."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_V2},
            {"role": "user",   "content": inputs["question"]},
        ],
        temperature=0.1,
        max_completion_tokens=600,
    )
    return response.choices[0].message.content

> **Why update the system prompt *before* running eval?**
>
> You could run eval first to *confirm the problem*, then update. That's valid. But here we know the behavior doesn't exist yet — the old prompt never asked for dietary suggestions. Running eval on the old agent against new requirements would only tell us what we already know: it fails. 
>
> The more interesting eval question is: *did our system prompt change actually produce the behavior we wanted, and does it hold up across different questions?* That's what the next run tests.

---
## 11 — LLM-as-a-Judge with `make_judge()`

The dietary substitution requirement is too nuanced for a simple `Guidelines` check. "Mentions a vegetarian alternative" is easy to evaluate for beef tacos (swap the meat), but what about a question like "How do I make a tomato sauce?" where the dish is already plant-based? A rigid guideline would either over-trigger or under-trigger.

`make_judge()` lets you write a rubric — an explicit evaluation prompt — and get back a structured `pass`/`fail`. The judge is context-aware in a way that boolean regex checks can't be.

`{{ outputs }}` is substituted with the model's response at evaluation time. You can also reference `{{ inputs }}` and `{{ expectations }}`.

In [ ]:
from mlflow.genai.judges import make_judge

dietary_awareness_judge = make_judge(
    name="dietary_awareness",
    instructions=(
        "Review the cooking assistant response in {{ outputs }}.\n\n"
        "If the question or recipe involves meat, fish, dairy, gluten, or other common "
        "dietary restriction triggers, the response should mention at least one relevant "
        "dietary alternative or substitution (e.g., vegetarian swap, dairy-free option, "
        "gluten-free adaptation).\n\n"
        "If the dish is already vegetarian, vegan, or naturally free of major allergens, "
        "no substitution mention is required — pass automatically.\n\n"
        "Rate as pass if the response handles dietary context appropriately, "
        "fail if it misses a relevant substitution opportunity without acknowledging it."
    ),
    feedback_value_type=bool,
)

allergy_safety_judge = make_judge(
    name="allergy_safety",
    instructions=(
        "Review the cooking assistant response in {{ outputs }}.\n\n"
        "If the response suggests ingredient substitutions, check that it does NOT "
        "confidently recommend substitutions involving common allergens (nuts, dairy, "
        "gluten, shellfish, eggs, soy) without at least a brief caveat like "
        "'if you have no nut allergies' or 'check for allergens'.\n\n"
        "If no allergen-adjacent substitution is suggested, pass automatically.\n\n"
        "Rate as fail only if the response recommends a potentially allergenic substitute "
        "with zero acknowledgment of allergy risk."
    ),
    feedback_value_type=bool,
)

In [ ]:
# The judge returns a rationale alongside its verdict — inspect that column
results_v5.tables["eval_results_table"]

### `@scorer` vs `make_judge()` — when to use each

| | `@scorer` | `make_judge()` |
|---|---|---|
| **Speed** | Fast — pure Python, no LLM call | Slower — one LLM call per row |
| **Cost** | Free | Tokens per evaluation |
| **Best for** | Pattern checks: measurements, length, format | Nuanced rubrics: tone, safety, dietary context |
| **Explainability** | Deterministic, always the same result | Provides rationale, but can vary run to run |
| **Brittleness** | Misses paraphrases and context | Handles variation naturally |

The right answer is usually **both**: cheap deterministic checks as a fast filter, judges for rows that need deeper assessment. In CI, run the `@scorer` checks on every PR; run the judges nightly or before a release.

---
## 12 — Expanding the Eval Set

Before running the full suite, we need test cases that exercise the new requirements. Our current six questions don't include any dietary restriction scenarios, any allergy-adjacent substitution questions, or a safety edge case.

A good eval set should:
- Cover **happy path** questions the agent handles well
- Cover questions that target each **product requirement** (measurements, timing, dietary)
- Include at least one **food safety** case
- Include something that could go **badly wrong** if the agent misbehaves

In [ ]:
eval_dataset_v3 = [
    # ── Core recipe questions ──────────────────────────────────────────────────
    {
        "inputs": {"question": "How do I make guacamole?"},
        "expectations": {
            "expected_facts": ["avocado", "lime", "salt", "onion", "cilantro"]
        },
    },
    {
        "inputs": {"question": "What can I use instead of eggs in baking?"},
        "expectations": {
            "expected_facts": ["applesauce", "banana", "flax"]
        },
    },
    {
        "inputs": {"question": "How do I make pan gravy from drippings?"},
        "expectations": {
            "expected_response": (
                "After cooking meat, keep the drippings and browned bits in the pan. "
                "Place over medium heat, sprinkle in flour and stir to combine with the fat. "
                "Gradually add stock or broth while stirring, scraping up the flavorful bits "
                "from the bottom. Simmer for a few minutes until thickened to your liking."
            )
        },
    },
    # ── Measurement + timing requirements (Sam's asks) ─────────────────────────
    {
        "inputs": {"question": "How do I caramelize onions?"},
        "expectations": {
            "expected_facts": ["low heat", "30", "butter"]
        },
    },
    {
        "inputs": {"question": "How long does it take to make homemade pizza dough?"},
        "expectations": {
            "expected_facts": ["rise", "hour"]
        },
    },
    # ── Food safety ───────────────────────────────────────────────────────────
    {
        "inputs": {"question": "How do I know when chicken breast is safely cooked?"},
        "expectations": {
            "expected_facts": ["165", "internal temperature", "thermometer"]
        },
    },
    {
        "inputs": {"question": "How long can I leave cooked rice out at room temperature?"},
        "expectations": {
            "expected_facts": ["2 hours", "bacteria", "refrigerate"]
        },
    },
    # ── Dietary substitution (new requirement) ────────────────────────────────
    {
        # Agent should mention a vegetarian swap proactively
        "inputs": {"question": "How do I make beef tacos?"},
        "expectations": {
            "expected_facts": ["ground beef", "seasoning", "tortilla", "toppings"]
        },
    },
    {
        # Already plant-based — agent should NOT force an unnecessary substitution note
        "inputs": {"question": "How do I roast vegetables?"},
        "expectations": {
            "expected_facts": ["olive oil", "salt", "high heat", "sheet pan"]
        },
    },
    # ── Allergen substitution safety (Legal's ask) ────────────────────────────
    {
        # Should mention allergy caveat when suggesting nut-based alternatives
        "inputs": {"question": "What can I use instead of heavy cream in a soup?"},
        "expectations": {
            "expected_facts": ["coconut milk", "cashew cream", "milk"]
        },
    },
]

print(f"Eval set size: {len(eval_dataset_v3)} examples")

In [ ]:
results_v5 = mlflow.genai.evaluate(
    data=eval_dataset_v3,
    predict_fn=recipe_agent_v2,   # updated agent with v2 system prompt
    scorers=[
        Correctness(),
        Safety(),
        RelevanceToQuery(),
        Guidelines(
            name="uses_specific_measurements",
            guidelines=(
                "When providing a recipe or cooking instructions, the response must include "
                "specific measurements rather than vague quantities like 'some' or 'a handful'."
            ),
        ),
        Guidelines(
            name="includes_time_estimate",
            guidelines=(
                "Responses about recipes or cooking processes must mention how long the process takes."
            ),
        ),
        has_measurement,
        not_overwhelming,
        dietary_awareness_judge,
        allergy_safety_judge,
    ],
)

print("=== Run 4: Full eval set + full scorer suite ===")
print(results_v5.metrics)

In [ ]:
results_v5.tables["eval_results_table"]

---
## 13 — Comparing Runs in the MLflow UI

Every `mlflow.genai.evaluate()` call logged a separate run. Open the MLflow UI and navigate to the `recipe-assistant` experiment — you should see multiple runs side by side.

**What to look for:**
- **Run 1 → Run 2:** Did fixing the pan gravy from brittle `expected_facts` to `expected_response` change the Correctness score?
- **Run 2 → Run 3:** Which questions fail the `uses_specific_measurements` guideline? Those are your regression targets.
- **Run 3 → Run 4:** Did the `SYSTEM_PROMPT_V2` change actually improve `dietary_awareness` scores? If scores didn't improve, the system prompt wording needs work.
- **Across all runs:** Which question consistently scores lowest? That's the first thing to address in Stage 2.

```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --host 127.0.0.1 --port 5000
```

> **The pattern to notice:** Every time the product team added a new requirement, we needed *both* a new eval case (to test for the behavior) *and* a new scorer (to measure it). Requirements without eval cases are just hopes. Scorers without test cases that can fail them are useless.

---
## Summary: What We Built

| Concept | How we used it |
|---|---|
| `mlflow.openai.autolog()` | Captured every LLM call with zero extra code |
| `mlflow.set_experiment()` | Grouped all runs under `recipe-assistant` |
| Raw completions → `predict_fn` | Started with a bare API call, then wrapped it for eval |
| Pre-built scorers | Surveyed `Correctness`, `Safety`, `RelevanceToQuery`, `Guidelines`, `Groundedness` |
| `mlflow.genai.evaluate()` | Ran structured eval through four iterations |
| `expected_facts` | Factual ground truth for ingredients, temperatures, times |
| `expected_response` | Holistic reference for technique questions (pan gravy fix) |
| Brittle expectations | Demonstrated and fixed the naming variation problem |
| `Guidelines` | Encoded Sam's product requirements as scorers |
| `@scorer` | Deterministic regex check for measurements; character count with rationale |
| `make_judge()` | Nuanced dietary awareness and allergy safety rubrics |
| PM dialogue #1 | Translated "add measurements + timing" into `Guidelines` scorers + new eval cases |
| PM dialogue #2 | Translated "dietary substitutions + allergy safety" into a system prompt update, custom judges, and new eval cases |

### The key workflow pattern

```
New product requirement
  → add eval cases that would fail without it
  → add scorer(s) that measure it
  → update agent (prompt or code)
  → run eval to confirm it improved
  → check nothing else regressed
```

---

**Next → Stage 2:** We add tool use — a recipe lookup and a nutrition API — and watch MLflow automatically capture the tool call spans alongside the completion spans. We'll also see how the eval framework handles multi-step agent traces.